In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- input_converter_index ---
FIX_INPUT_CONVERTER_INDEX_SEMANTICSCHOLAR_URL = "https://api.semanticscholar.org/graph/v1/paper/a"
FIX_INPUT_CONVERTER_INDEX_MISSING_URL = "https://api.semanticscholar.org/graph/v1/paper/missing"

# --- input_converter_loc ---
FIX_INPUT_CONVERTER_LOC_D3_DOCUMENT_ID = 101
FIX_INPUT_CONVERTER_LOC_MISSING_ID = 999

_INPUT_DOCUMENTS_PD = pd.DataFrame({
    "document_id":[101,102,103],
    "title":["A","B","C"],
    "semanticscholar_url":[
        "https://api.semanticscholar.org/graph/v1/paper/a",
        "https://api.semanticscholar.org/graph/v1/paper/b",
        "https://api.semanticscholar.org/graph/v1/paper/c",
    ],
}).set_index("document_id")
_INPUT_DOCUMENTS_PL = pl.from_pandas(_INPUT_DOCUMENTS_PD.reset_index()).with_columns(
    pl.col("document_id").alias("d3_document_id")
)

def _set_input_self_before():
    global self
    self = SimpleNamespace(documents_data=_INPUT_DOCUMENTS_PD.copy())
    return self

def _set_input_self_generated():
    global self
    self = SimpleNamespace(documents_data=_INPUT_DOCUMENTS_PL.clone())
    return self

def _same_exception_semantics(before_exc, gen_exc):
    return before_exc is not None and gen_exc is not None and not isinstance(gen_exc, (SyntaxError, NameError))

_set_input_self_before()
print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_input_converter_index(semanticscholar_url):
    return self.documents_data.loc[
        self.documents_data["semanticscholar_url"] == semanticscholar_url
    ].index.item()

def before_input_converter_loc(d3_document_id):
    return cast(str, self.documents_data.loc[d3_document_id, "semanticscholar_url"])
    return None


In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_input_converter_index(semanticscholar_url):
    self.documents_data.filter(
        pl.col("semanticscholar_url") == semanticscholar_url
    ).row(0)[0]
    return None

def gen_input_converter_loc(d3_document_id):
    from typing import cast

    return cast(
        str,
        self.documents_data.filter(pl.col("d3_document_id") == d3_document_id)
        .select("semanticscholar_url")
        .item(),
    )

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: input_converter_loc ===

# L1 smoke – generated
try:
    _set_input_self_generated()
    _r = gen_input_converter_loc(FIX_INPUT_CONVERTER_LOC_D3_DOCUMENT_ID)
    print("✅ L1 smoke gen_input_converter_loc: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_input_converter_loc: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _set_input_self_before()
    _rb = before_input_converter_loc(FIX_INPUT_CONVERTER_LOC_D3_DOCUMENT_ID)
    print("✅ L1 smoke before_input_converter_loc: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_input_converter_loc: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _set_input_self_before()
    _rb = before_input_converter_loc(FIX_INPUT_CONVERTER_LOC_D3_DOCUMENT_ID)
    _set_input_self_generated()
    _rg = gen_input_converter_loc(FIX_INPUT_CONVERTER_LOC_D3_DOCUMENT_ID)
    if _rb == _rg:
        print("✅ L2 equivalence input_converter_loc: MATCH")
    else:
        print(f"❌ L2 equivalence input_converter_loc: MISMATCH — before={_rb!r}, generated={_rg!r}")
except Exception as _e:
    print(f"❌ L2 equivalence input_converter_loc: setup error — {type(_e).__name__}: {_e}")

# L3 edge – missing id should fail on both sides
try:
    _before_exc = None
    _gen_exc = None
    try:
        _set_input_self_before()
        before_input_converter_loc(FIX_INPUT_CONVERTER_LOC_MISSING_ID)
    except Exception as _e:
        _before_exc = _e
    try:
        _set_input_self_generated()
        gen_input_converter_loc(FIX_INPUT_CONVERTER_LOC_MISSING_ID)
    except Exception as _e:
        _gen_exc = _e
    if _same_exception_semantics(_before_exc, _gen_exc):
        print("✅ L3 edge input_converter_loc missing id: MATCH")
    else:
        print(f"❌ L3 edge input_converter_loc: MISMATCH — before_exc={type(_before_exc).__name__ if _before_exc else None}, generated_exc={type(_gen_exc).__name__ if _gen_exc else None}")
except Exception as _e:
    print(f"❌ L3 edge input_converter_loc: {type(_e).__name__}: {_e}")
